In [2]:
!pip install pyspark

# Introdução ao Apache Spark

## O que é Apache Spark?

Apache Spark é um framework de processamento distribuído de código aberto projetado para computação rápida em grandes conjuntos de dados. Diferente do Hadoop MapReduce tradicional, o Spark processa dados em memória, o que o torna significativamente mais rápido para muitas aplicações.

### Principais características do Spark:

1. **Velocidade**: Execução em memória que pode ser até 100x mais rápida que o Hadoop para determinadas cargas de trabalho
2. **Facilidade de uso**: APIs em Python, Java, Scala e R
3. **Generalidade**: Suporta SQL, streaming, machine learning e processamento de grafos
4. **Execução em qualquer lugar**: Pode ser executado em Hadoop, Mesos, Kubernetes, standalone ou na nuvem

## Arquitetura do Spark

O Spark é baseado em dois conceitos principais:

1. **RDD (Resilient Distributed Dataset)**: Coleção distribuída de elementos que pode ser processada em paralelo
2. **DataFrame/Dataset**: APIs estruturadas de alto nível construídas sobre RDDs

A arquitetura do Spark inclui:
Componentes principais:

#### Driver Program:

Contém a aplicação principal e cria o SparkContext.
Responsável por converter o código do usuário em tarefas que serão executadas pelos executores.
Orquestra o trabalho distribuído e gerencia o estado da aplicação.


#### SparkContext:

Ponto de entrada para qualquer funcionalidade Spark.
Representa a conexão com o cluster Spark.
Responsável por configurar serviços internos e estabelecer conexão com o gerenciador de cluster.
Em PySpark moderno, muitas vezes substituído pelo objeto SparkSession que encapsula SparkContext.


#### Cluster Manager:

Gerencia recursos no cluster (YARN, Mesos, Kubernetes ou gerenciador standalone do Spark).
Aloca recursos e monitora a execução dos aplicativos Spark.


#### Executors:

Processos JVM iniciados em nós de trabalho.
Executam as tarefas atribuídas pelo driver.
Armazenam dados em cache quando solicitado.
Reportam resultados de volta ao driver.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [5]:
# Configuração otimizada
spark = SparkSession.builder \
    .appName("Spark Avançado") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "400s") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "10").getOrCreate()

In [6]:
# Verificando a versão do Spark
print(f"Versão do Spark: {spark.version}")

Versão do Spark: 3.5.1


In [7]:
file_path ="/content/sales.csv"

## importar o csv

In [8]:
# Carregando o arquivo CSV como um DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

In [9]:
# Exibindo o schema do DataFrame
print("Schema do DataFrame:")
df.printSchema()


Schema do DataFrame:
root
 |-- Data: date (nullable = true)
 |-- ID_Cliente: string (nullable = true)
 |-- Nome_Cliente: string (nullable = true)
 |-- ID_Vendedor: string (nullable = true)
 |-- Nome_Vendedor: string (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Categoria_Produto: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Quantidade: integer (nullable = true)
 |-- Preco_Unitario: integer (nullable = true)
 |-- Desconto: double (nullable = true)
 |-- Valor_Total: double (nullable = true)
 |-- Custo: integer (nullable = true)
 |-- Lucro: double (nullable = true)
 |-- Metodo_Envio: string (nullable = true)
 |-- Frequencia_Compra: integer (nullable = true)



## Colunas derivadas

In [11]:
df_transformado = df.withColumn("faixa_preco",
                              F.when(F.col("Valor_Total") < 3000, "Baixo" )
                              .when((F.col("Valor_Total") >= 3000) & (F.col("Valor_Total") < 10000), "Médio")
                              .otherwise("Alto"))



In [12]:
df_transformado.show()

+----------+----------+----------------+-----------+---------------+--------------+------+-----------------+--------------+----------+--------------+--------+-----------+-----+-----+------------+-----------------+-----------+
|      Data|ID_Cliente|    Nome_Cliente|ID_Vendedor|  Nome_Vendedor|        Cidade|Estado|Categoria_Produto|       Produto|Quantidade|Preco_Unitario|Desconto|Valor_Total|Custo|Lucro|Metodo_Envio|Frequencia_Compra|faixa_preco|
+----------+----------+----------------+-----------+---------------+--------------+------+-----------------+--------------+----------+--------------+--------+-----------+-----+-----+------------+-----------------+-----------+
|2023-01-05|      C001|      João Silva|       V001|   Ana Oliveira|     São Paulo|    SP|      Eletrônicos|    Smartphone|         2|          1200|    0.05|     2280.0| 1800|480.0|    Expresso|                3|      Baixo|
|2023-01-10|      C002|    Maria Santos|       V002|    Bruno Costa|Rio de Janeiro|    RJ|      

## Agregação Avançadas

In [13]:
resumo  = df.groupBy("Cidade") \
      .agg(F.sum("Valor_Total").alias("Total_Vendas"),
           F.avg("Valor_Total").alias("Media_Vendas"),
           F.max("Valor_Total").alias("Maximo_Vendas"),
           F.min("Valor_Total").alias("Minimo_Vendas"),
           F.count("*").alias("Quantidade_Vendas"),
           F.round(F.stddev("Valor_Total"),2).alias("Desvio_Padrao_Vendas"))

In [14]:
resumo.show()

+--------------+------------+------------------+-------------+-------------+-----------------+--------------------+
|        Cidade|Total_Vendas|      Media_Vendas|Maximo_Vendas|Minimo_Vendas|Quantidade_Vendas|Desvio_Padrao_Vendas|
+--------------+------------+------------------+-------------+-------------+-----------------+--------------------+
|Rio de Janeiro|     14445.0|          1805.625|       3420.0|        650.0|                8|              916.31|
|  Porto Alegre|      1926.0|             481.5|        950.0|         70.0|                4|               380.2|
|Belo Horizonte|     10904.0|1557.7142857142858|       3500.0|        114.0|                7|             1415.13|
|      Curitiba|      1832.0|             458.0|        850.0|        300.0|                4|              262.05|
|     São Paulo|      9523.5|         1190.4375|       2280.0|        351.0|                8|              747.74|
|      Salvador|       827.6|             206.9|        360.0|         8

### JOINS

In [16]:
# Criando um segundo DataFrame para joins
dados_projetos = [
    ("projeto1", "CRM", "2023-01-10", "2023-05-15"),
    ("projeto2", "ERP", "2022-08-20", "2023-02-28"),
    ("projeto3", "App Mobile", "2023-03-05", "2023-07-20"),
    ("projeto4", "Dashboard", "2023-02-15", "2023-04-30"),
    ("projeto5", "API", "2023-04-10", None)
]

schema_projetos = ["id_projeto", "nome_projeto", "data_inicio", "data_fim"]
df_projetos = spark.createDataFrame(dados_projetos, schema_projetos)
df_projetos.show()

+----------+------------+-----------+----------+
|id_projeto|nome_projeto|data_inicio|  data_fim|
+----------+------------+-----------+----------+
|  projeto1|         CRM| 2023-01-10|2023-05-15|
|  projeto2|         ERP| 2022-08-20|2023-02-28|
|  projeto3|  App Mobile| 2023-03-05|2023-07-20|
|  projeto4|   Dashboard| 2023-02-15|2023-04-30|
|  projeto5|         API| 2023-04-10|      NULL|
+----------+------------+-----------+----------+



In [17]:
dados_responsaveis = [
    ("projeto1", "Carlos"),
    ("projeto2", "Maria"),
    ("projeto3", "Ana"),
    ("projeto6", "João")  # Este projeto não existe no df_projetos
]

schema_responsaveis = ["id_projeto", "responsavel"]

df_responsaveis = spark.createDataFrame(dados_responsaveis, schema_responsaveis)
df_responsaveis.show()

+----------+-----------+
|id_projeto|responsavel|
+----------+-----------+
|  projeto1|     Carlos|
|  projeto2|      Maria|
|  projeto3|        Ana|
|  projeto6|       João|
+----------+-----------+



In [20]:
df_projetos.cache()
df_responsaveis.cache()
df_projetos.count()
df_responsaveis.count()

4

## Inner Join (Só traz os correspondentes)

In [21]:
df_projetos.join(df_responsaveis, "id_projeto", "inner").show()

+----------+------------+-----------+----------+-----------+
|id_projeto|nome_projeto|data_inicio|  data_fim|responsavel|
+----------+------------+-----------+----------+-----------+
|  projeto1|         CRM| 2023-01-10|2023-05-15|     Carlos|
|  projeto2|         ERP| 2022-08-20|2023-02-28|      Maria|
|  projeto3|  App Mobile| 2023-03-05|2023-07-20|        Ana|
+----------+------------+-----------+----------+-----------+



## Lef Join

In [22]:
df_projetos.join(df_responsaveis, "id_projeto", "left").show()

+----------+------------+-----------+----------+-----------+
|id_projeto|nome_projeto|data_inicio|  data_fim|responsavel|
+----------+------------+-----------+----------+-----------+
|  projeto1|         CRM| 2023-01-10|2023-05-15|     Carlos|
|  projeto2|         ERP| 2022-08-20|2023-02-28|      Maria|
|  projeto3|  App Mobile| 2023-03-05|2023-07-20|        Ana|
|  projeto4|   Dashboard| 2023-02-15|2023-04-30|       NULL|
|  projeto5|         API| 2023-04-10|      NULL|       NULL|
+----------+------------+-----------+----------+-----------+



## Right JOIn

In [23]:
df_projetos.join(df_responsaveis, "id_projeto", "right").show()

+----------+------------+-----------+----------+-----------+
|id_projeto|nome_projeto|data_inicio|  data_fim|responsavel|
+----------+------------+-----------+----------+-----------+
|  projeto1|         CRM| 2023-01-10|2023-05-15|     Carlos|
|  projeto2|         ERP| 2022-08-20|2023-02-28|      Maria|
|  projeto3|  App Mobile| 2023-03-05|2023-07-20|        Ana|
|  projeto6|        NULL|       NULL|      NULL|       João|
+----------+------------+-----------+----------+-----------+

